In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import IsolationForest

root = Path(r"D:\ResistMap-AI")
table = pd.read_csv(
    root / "data/processed/klebsiella_amr_features.csv"
)

gene_columns = [
    c for c in table.columns if c.startswith("gene__")
]
genes = table[gene_columns]

# Validate inputs before fitting.
assert gene_columns, "Gene columns were not found."
assert table["Isolate"].notna().all(), "Missing isolate IDs."
assert table["Isolate"].is_unique, "Duplicate isolate IDs."
assert genes.isin([0, 1]).all().all(), "Non-binary/missing gene values."

binary = genes.to_numpy(dtype=np.uint8)

# Identical profiles stay together across the three partitions.
profiles = [row.tobytes() for row in binary]
groups = pd.factorize(
         pd.Series(profiles, dtype="object"),
         sort=False
)[0]

if len(np.unique(groups)) < 10:
    raise ValueError("Too few distinct profiles for this evaluation.")

splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.20, random_state=42
)
development_idx, test_idx = next(
    splitter.split(binary, groups=groups)
)

splitter_cal = GroupShuffleSplit(
    n_splits=1, test_size=0.25, random_state=43
)
train_local, cal_local = next(
    splitter_cal.split(
        binary[development_idx],
        groups=groups[development_idx]
    )
)

train_idx = development_idx[train_local]
cal_idx = development_idx[cal_local]

for first, second in [
    (train_idx, cal_idx),
    (train_idx, test_idx),
    (cal_idx, test_idx),
]:
    assert not set(groups[first]) & set(groups[second])

# Remove only columns constant in TRAINING.
# Retain rare genes that vary in training.
training_frequency = binary[train_idx].mean(axis=0)
keep = (training_frequency > 0) & (training_frequency < 1)

if not keep.any():
    raise ValueError("No variable training features.")

features = binary[:, keep].astype(np.float32)

validation_model = IsolationForest(
    n_estimators=300,
    max_samples=min(256, len(train_idx)),
    contamination="auto",
    random_state=42,
    n_jobs=-1
)
validation_model.fit(features[train_idx])

# Threshold chosen on calibration data only.
# 3% is a review-budget choice, NOT a measured anomaly rate.
cal_scores = -validation_model.score_samples(features[cal_idx])
threshold = float(np.quantile(cal_scores, 0.97))

test_scores = -validation_model.score_samples(features[test_idx])

evaluation = table.iloc[test_idx][[
    "Isolate", "Scientific name",
    "Collection date", "Location"
]].copy()

evaluation["complete_gene_count"] = binary[test_idx].sum(axis=1)
evaluation["anomaly_score"] = test_scores
evaluation["review_flag"] = test_scores > threshold

# Genes absent from training cannot be learned by this model.
unseen_columns = training_frequency == 0
evaluation["genes_unseen_in_training"] = (
    binary[test_idx][:, unseen_columns].sum(axis=1)
)

print(
    "Rows — training:", len(train_idx),
    "| calibration:", len(cal_idx),
    "| evaluation:", len(test_idx)
)
print("Training gene features:", int(keep.sum()))
print("Identical-profile overlap between partitions: 0")
print("Evaluation samples flagged:",
      int(evaluation["review_flag"].sum()))
print("Evaluation flag rate:",
      f"{evaluation['review_flag'].mean():.1%}")

# Diagnostic: does the score still track gene count?
rho = evaluation["anomaly_score"].rank().corr(
    evaluation["complete_gene_count"].rank()
)
print("Score–gene count rank correlation:", round(rho, 3))

display(
    evaluation.groupby("review_flag")["complete_gene_count"]
    .agg(["count", "median", "mean"]).round(2)
)
display(
    evaluation.sort_values("anomaly_score", ascending=False).head(10)
)

results_dir = root / "results"
results_dir.mkdir(parents=True, exist_ok=True)
evaluation.to_csv(
    results_dir / "heldout_anomaly_review.csv", index=False
)

Rows — training: 1793 | calibration: 578 | evaluation: 629
Training gene features: 354
Identical-profile overlap between partitions: 0
Evaluation samples flagged: 31
Evaluation flag rate: 4.9%
Score–gene count rank correlation: 0.909


,count,median,mean
review_flag,,,
False,598,11.0,11.76
True,31,24.0,25.55


,Isolate,Scientific name,Collection date,Location,complete_gene_count,anomaly_score,review_flag,genes_unseen_in_training
891,PDT003141248.1,Klebsiella pneumoniae,2023-08-31,"China: Guangdong, Yunfu",34,0.525600,True,0
592,PDT003062392.1,Klebsiella pneumoniae,2017-04-12,Kenya,32,0.519588,True,0
453,PDT003000998.1,Klebsiella pneumoniae,2024-10-30,China,34,0.488363,True,1
758,PDT003267369.1,Klebsiella pneumoniae,2019,Thailand:Roi Et,30,0.483280,True,0
725,PDT001095136.1,Klebsiella pneumoniae,NaN,UAE,34,0.475092,True,1
1519,PDT000885015.1,Klebsiella pneumoniae,NaN,NaN,31,0.469751,True,1
2735,PDT000431973.1,Klebsiella pneumoniae,2016,Viet Nam:Hanoi,26,0.464924,True,0
2086,PDT001307931.1,Klebsiella pneumoniae,NaN,Singapore,29,0.456433,True,0
1964,PDT001816822.2,Klebsiella pneumoniae,2023,USA,20,0.452158,True,0
2638,PDT002793982.1,Klebsiella pneumoniae,2023-05-19,Latvia,23,0.449511,True,0
